# 目标检测与语义分割模型

本notebook介绍现代目标检测和语义分割的经典模型。

## 学习目标

- 理解SSD单阶段检测器
- 掌握R-CNN系列两阶段检测器
- 学习语义分割和FCN模型
- 了解转置卷积的作用

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchvision import transforms
import matplotlib.pyplot as plt
import numpy as np

print(f"PyTorch版本: {torch.__version__}")
print(f"Torchvision版本: {torchvision.__version__}")

## 第一部分: SSD (Single Shot MultiBox Detector)

### 1.1 SSD概述

**特点**:
- **单阶段 (Single Shot)**: 一次前向传播完成检测
- **多尺度检测**: 在多个特征图上预测
- **速度快**: 实时检测(30+ FPS)

**vs 两阶段检测器**:
```
SSD (单阶段):
  图像 → CNN → 多尺度特征图 → 直接预测
  优点: 快速
  缺点: 精度略低

R-CNN (两阶段):
  图像 → 提议区域 → CNN特征 → 分类+回归
  优点: 精度高
  缺点: 较慢
```

### 1.2 SSD架构

```
输入图像 (300×300)
    ↓
┌─────────────────────┐
│ 基础网络 (VGG-16)   │ ← 预训练特征提取
│ Conv1-5             │
└──────┬──────────────┘
       │
       ├→ 特征图1 (38×38) → 检测小目标
       │      ↓
       ├→ 特征图2 (19×19) → 检测中目标
       │      ↓
       ├→ 特征图3 (10×10) → 检测大目标
       │      ↓
       ├→ 特征图4 (5×5)   → 检测超大目标
       │      ↓
       └→ 特征图5 (3×3)   → 检测极大目标
              ↓
       每个特征图预测:
       - 类别分数 (q+1类)
       - 边界框偏移 (4个值)
```

**关键思想**: 多尺度特征金字塔
- 大特征图(浅层): 小感受野 → 检测小目标
- 小特征图(深层): 大感受野 → 检测大目标

### 1.3 SSD核心组件

In [ ]:
# 类别预测层
def cls_predictor(num_inputs, num_anchors, num_classes):
    """预测每个锚框的类别
    
    Args:
        num_inputs: 输入通道数
        num_anchors: 每个位置的锚框数
        num_classes: 类别数(不含背景)
    
    Returns:
        3×3卷积层, 输出 num_anchors*(num_classes+1) 个通道
    """
    return nn.Conv2d(num_inputs, num_anchors * (num_classes + 1),
                    kernel_size=3, padding=1)

# 边界框预测层
def bbox_predictor(num_inputs, num_anchors):
    """预测每个锚框的偏移量
    
    Args:
        num_inputs: 输入通道数
        num_anchors: 每个位置的锚框数
    
    Returns:
        3×3卷积层, 输出 num_anchors*4 个通道
    """
    return nn.Conv2d(num_inputs, num_anchors * 4,
                    kernel_size=3, padding=1)

# 测试预测层
print("=== SSD预测层 ===")
num_classes = 10  # 如CIFAR-10
feature_map = torch.zeros((2, 256, 20, 20))  # batch=2, channels=256, 20×20

cls_pred = cls_predictor(256, num_anchors=5, num_classes=num_classes)
bbox_pred = bbox_predictor(256, num_anchors=5)

cls_output = cls_pred(feature_map)
bbox_output = bbox_pred(feature_map)

print(f"特征图: {feature_map.shape}")
print(f"类别预测: {cls_output.shape}")
print(f"  - {cls_output.shape[1]} = 5锚框 × {num_classes+1}类")
print(f"边界框预测: {bbox_output.shape}")
print(f"  - {bbox_output.shape[1]} = 5锚框 × 4坐标")

print("\n理解:")
print("  - 每个像素: 5个锚框")
print("  - 总锚框数: 20×20×5 = 2000")
print(f"  - 每个锚框: {num_classes+1}个类别分数 + 4个坐标")

### 1.4 多尺度预测连接

In [ ]:
def flatten_pred(pred):
    """展平预测结果以便连接
    
    Args:
        pred: shape (batch, C, H, W)
    
    Returns:
        shape (batch, H*W*C)
    """
    return torch.flatten(pred.permute(0, 2, 3, 1), start_dim=1)

def concat_preds(preds):
    """连接多个尺度的预测
    
    Args:
        preds: list of predictions from different scales
    
    Returns:
        concatenated predictions along dimension 1
    """
    return torch.cat([flatten_pred(p) for p in preds], dim=1)

# 演示多尺度连接
print("=== 多尺度预测连接 ===")
# 模拟两个不同尺度的预测
Y1 = torch.zeros((2, 55, 20, 20))  # 尺度1: 20×20, 5锚框×11类
Y2 = torch.zeros((2, 33, 10, 10))  # 尺度2: 10×10, 3锚框×11类

print(f"尺度1预测: {Y1.shape}")
print(f"尺度2预测: {Y2.shape}")

# 展平
Y1_flat = flatten_pred(Y1)
Y2_flat = flatten_pred(Y2)
print(f"\n展平后:")
print(f"  尺度1: {Y1_flat.shape}  # 20×20×55 = {20*20*55}")
print(f"  尺度2: {Y2_flat.shape}  # 10×10×33 = {10*10*33}")

# 连接
combined = concat_preds([Y1, Y2])
print(f"\n连接后: {combined.shape}")
print(f"  总预测数: {20*20*55 + 10*10*33} = {combined.shape[1]}")

print("\n意义:")
print("  - 不同尺度特征图有不同大小")
print("  - 展平并连接 → 统一格式")
print("  - 方便后续处理(NMS等)")

### 1.5 下采样块

In [ ]:
def down_sample_blk(in_channels, out_channels):
    """下采样块: 减小特征图尺寸
    
    结构:
      Conv1(3×3) → ReLU → Conv2(3×3) → ReLU → MaxPool(2×2, stride=2)
    
    效果:
      - 高宽减半
      - 感受野扩大
    """
    blk = []
    for _ in range(2):
        blk.append(nn.Conv2d(in_channels, out_channels,
                            kernel_size=3, padding=1))
        blk.append(nn.ReLU())
        in_channels = out_channels
    blk.append(nn.MaxPool2d(kernel_size=2, stride=2))
    return nn.Sequential(*blk)

# 测试下采样
print("=== 下采样块 ===")
blk = down_sample_blk(3, 10)
X = torch.zeros((2, 3, 20, 20))
Y = blk(X)

print(f"输入: {X.shape}")
print(f"输出: {Y.shape}")
print(f"\n变化:")
print(f"  - 通道: 3 → 10")
print(f"  - 空间: 20×20 → 10×10 (减半)")

print("\n感受野分析:")
print("  - 每个3×3卷积: 感受野+2")
print("  - 2×2池化(stride=2): 感受野×2")
print("  - 总效果: 输出1个单元 ≈ 输入6×6区域")

### 1.6 SSD总结

**优点**:
- ✅ **速度快**: 单次前向传播,实时检测
- ✅ **多尺度**: 不同层检测不同大小目标
- ✅ **端到端**: 直接优化,无需区域提议

**缺点**:
- ❌ **小目标**: 检测较困难(特征图太小)
- ❌ **精度**: 略低于两阶段方法
- ❌ **密集目标**: 重叠严重时效果差

**应用场景**:
- 实时视频监控
- 自动驾驶
- 移动设备检测

**后续改进**:
- YOLOv1-v8: 更快更准
- RetinaNet: Focal Loss解决类别不平衡
- EfficientDet: 模型缩放策略

## 第二部分: R-CNN系列

### 2.1 目标检测发展史

```
传统方法 (2012前):
  滑动窗口 + 手工特征(HOG, SIFT) + SVM
  ↓
R-CNN (2014):
  选择性搜索 + CNN特征 + SVM
  ↓
Fast R-CNN (2015):
  RoI池化 + 端到端训练
  ↓
Faster R-CNN (2015):
  RPN区域提议网络
  ↓
Mask R-CNN (2017):
  + 实例分割
```

### 2.2 R-CNN (2014)

**核心思想**: CNN用于目标检测

**流程**:
```
1. 选择性搜索 → 2000个候选区域
2. 每个区域 → 变形到227×227 → CNN(AlexNet)
3. CNN特征 → SVM分类器(每类一个)
4. CNN特征 → 线性回归(边界框精修)
```

**优点**:
- ✅ 首次成功应用CNN
- ✅ 大幅提升精度(mAP +30%)

**缺点**:
- ❌ **极慢**: 每张图2000次CNN前向传播
- ❌ **训练复杂**: 三阶段(CNN, SVM, 回归)
- ❌ **重复计算**: 区域重叠,特征重复提取

### 2.3 Fast R-CNN (2015)

**改进**: 共享卷积计算

**流程**:
```
1. 整张图 → CNN → 特征图
2. 选择性搜索 → 候选区域
3. RoI池化: 特征图 + 区域 → 固定大小特征
4. 全连接层 → 分类 + 回归
```

**RoI池化 (Region of Interest Pooling)**:
```
问题: 不同区域 → 不同特征大小
解决: 划分网格 → 每格最大值 → 固定输出

示例:
  输入RoI: 7×5特征
  输出: 2×2
  
  划分4个子窗口:
  ┌────┬────┐
  │ 3.5│ 2.5│ → 取每格最大值
  │ ×3 │ ×3 │
  ├────┼────┤
  │ 3.5│ 2.5│
  │ ×2 │ ×2 │
  └────┴────┘
```

In [ ]:
# RoI池化演示
print("=== RoI池化 ===")
# 特征图
X = torch.arange(16.).reshape(1, 1, 4, 4)
print("特征图 (4×4):")
print(X.squeeze())

# 定义RoI: [batch_idx, x1, y1, x2, y2]
# 假设图像40×40, 特征图是1/10, 所以spatial_scale=0.1
rois = torch.Tensor([
    [0, 0, 0, 20, 20],   # 左上角RoI
    [0, 0, 10, 30, 30]   # 较大RoI
])

# RoI池化
output = torchvision.ops.roi_pool(X, rois, output_size=(2, 2), 
                                 spatial_scale=0.1)

print("\nRoI定义:")
print("  RoI1: (0,0,20,20) → 特征图(0,0,2,2)")
print("  RoI2: (0,10,30,30) → 特征图(0,1,3,3)")

print("\n池化输出 (2×2×2):")
for i in range(2):
    print(f"  RoI{i+1}:")
    print(output[i].squeeze())

print("\n理解:")
print("  - 不同大小的RoI")
print("  - 统一池化到2×2")
print("  - 每个子窗口取最大值")

### 2.4 Faster R-CNN (2015)

**改进**: RPN替代选择性搜索

**RPN (Region Proposal Network)**:
```
CNN特征图
    ↓
3×3卷积 → 每个位置的特征
    ↓
┌─────────┬─────────┐
│ 分类头  │ 回归头  │
│ 1×1卷积 │ 1×1卷积 │
└─────────┴─────────┘
    ↓         ↓
目标/背景  边界框偏移
```

**完整流程**:
```
图像 → CNN → 特征图
              ↓
          ┌───RPN───┐
          │ 生成提议│
          │ + NMS  │
          └────┬───┘
               ↓
          RoI池化
               ↓
          全连接层
               ↓
        ┌──────┴──────┐
     分类head      回归head
```

**优势**:
- ✅ **端到端**: RPN可学习,无需外部算法
- ✅ **快速**: 共享卷积,提议生成快
- ✅ **高质量**: 学习到的提议更好

**训练**:
- 联合优化: RPN损失 + 检测损失
- 交替训练 or 近似联合训练

### 2.5 Mask R-CNN (2017)

**扩展**: 目标检测 + 实例分割

**新增**:
```
Faster R-CNN
     +
分割分支 (FCN)
     ↓
像素级掩码
```

**改进**:
1. **RoI Align** (替代RoI Pooling):
   - 问题: RoI池化会量化坐标(不对齐)
   - 解决: 双线性插值,保留空间精度
   - 效果: 像素级预测更准确

2. **分割头**:
   - 全卷积网络(FCN)
   - 每个RoI预测28×28掩码
   - 上采样到原始大小

**应用**:
- 实例分割(COCO数据集)
- 人体姿态估计
- 全景分割

### 2.6 R-CNN系列对比

| 模型 | 提议方法 | 特征提取 | 速度(FPS) | mAP | 年份 |
|------|----------|----------|-----------|-----|------|
| R-CNN | 选择性搜索 | 每个RoI一次 | 0.02 | 66% | 2014 |
| Fast R-CNN | 选择性搜索 | 共享卷积 | 0.5 | 68% | 2015 |
| Faster R-CNN | RPN | 共享卷积 | 7 | 73% | 2015 |
| Mask R-CNN | RPN | 共享卷积+RoI Align | 5 | 76% | 2017 |

**演进趋势**:
```
速度: 0.02 → 7 FPS  (350倍提升)
精度: 66% → 76% mAP (10%提升)
功能: 检测 → 检测+分割
```

**选择建议**:
- **追求精度**: Mask R-CNN
- **平衡速度精度**: Faster R-CNN
- **实时检测**: SSD, YOLO
- **实例分割**: Mask R-CNN

## 第三部分: 语义分割

### 3.1 任务定义

**三种分割任务**:

**1. 图像分割 (Image Segmentation)**:
```
- 目标: 将图像分成若干区域
- 方法: 聚类,边缘检测(传统CV)
- 特点: 无语义,仅根据像素相似度
```

**2. 语义分割 (Semantic Segmentation)**:
```
- 目标: 每个像素分配类别标签
- 输出: 像素级分类图
- 特点: 同类目标不区分

示例:
  原图: 2只猫 + 1只狗
  输出: [猫像素, 猫像素, 狗像素]
       (不区分哪只猫)
```

**3. 实例分割 (Instance Segmentation)**:
```
- 目标: 区分不同实例
- 输出: 每个实例的掩码
- 特点: 语义 + 实例ID

示例:
  原图: 2只猫 + 1只狗
  输出: [猫1, 猫2, 狗1]
```

**对比**:
```
┌──────────┬──────────┬──────────┐
│ 原图     │ 语义分割 │ 实例分割 │
├──────────┼──────────┼──────────┤
│   🐱🐱   │   AA     │   12     │
│   🐶    │   B      │   3      │
└──────────┴──────────┴──────────┘
  A=猫, B=狗  1,2=两只猫, 3=狗
```

### 3.2 Pascal VOC2012数据集

**特点**:
- 21类: 背景 + 20个对象类
- 训练集: 1464张
- 验证集: 1449张
- 像素级标注

**类别**:
```
人: person
动物: bird, cat, cow, dog, horse, sheep
交通工具: aeroplane, bicycle, boat, bus, car, motorbike, train
室内: bottle, chair, dining table, potted plant, sofa, tv/monitor
```

**数据格式**:
```
图像: JPEGImages/*.jpg
标签: SegmentationClass/*.png
  - RGB颜色 → 类别
  - 白色: 边界(忽略)
  - 黑色: 背景
  - 其他: 对象类
```

In [ ]:
# VOC2012颜色映射
VOC_COLORMAP = [
    [0, 0, 0],        # 0: background
    [128, 0, 0],      # 1: aeroplane
    [0, 128, 0],      # 2: bicycle
    [128, 128, 0],    # 3: bird
    [0, 0, 128],      # 4: boat
    [128, 0, 128],    # 5: bottle
    [0, 128, 128],    # 6: bus
    [128, 128, 128],  # 7: car
    [64, 0, 0],       # 8: cat
    [192, 0, 0],      # 9: chair
    [64, 128, 0],     # 10: cow
    # ... 更多类别
]

VOC_CLASSES = [
    'background', 'aeroplane', 'bicycle', 'bird', 'boat',
    'bottle', 'bus', 'car', 'cat', 'chair', 'cow',
    'diningtable', 'dog', 'horse', 'motorbike', 'person',
    'potted plant', 'sheep', 'sofa', 'train', 'tv/monitor'
]

print("=== VOC2012数据集 ===")
print(f"类别数: {len(VOC_CLASSES)}")
print(f"\n前10类:")
for i, (name, color) in enumerate(zip(VOC_CLASSES[:10], VOC_COLORMAP[:10])):
    print(f"  {i}: {name:12s} - RGB{color}")

print("\n数据特点:")
print("  - 像素级标注(耗时!)")
print("  - RGB→类别索引映射")
print("  - 边界像素(白色)忽略")
print("  - 多目标场景")

## 第四部分: FCN (Fully Convolutional Network)

### 4.1 FCN核心思想

**传统CNN (分类)**:
```
图像(224×224) → Conv → Pool → ... → Flatten → FC → 类别
                                      ↓
                            丢失空间信息!
```

**FCN (分割)**:
```
图像(H×W) → Conv → ... → 1×1 Conv → 转置卷积 → 预测图(H×W)
                          ↓             ↓
                    调整通道数     恢复空间尺寸
                     (→类别数)
```

**关键创新**:
1. **全卷积**: 无全连接层,保留空间结构
2. **1×1卷积**: 调整通道数到类别数
3. **转置卷积**: 上采样恢复分辨率

### 4.2 转置卷积 (Transposed Convolution)

**问题**: 卷积/池化 → 特征图变小 → 如何恢复?

**解决**: 转置卷积 = 学习的上采样

**vs 其他上采样**:
```
最近邻插值: 简单复制,无学习
双线性插值: 固定权重插值
转置卷积: 可学习的上采样 ✅
```

**工作原理**:
```
正常卷积 (下采样):
  4×4 → [Conv 2×2, stride=2] → 2×2

转置卷积 (上采样):
  2×2 → [TransConv 2×2, stride=2] → 4×4

数学: 类似卷积的"反向传播"
```

**参数设计**:
```
要放大s倍:
  - stride = s
  - kernel_size = 2s
  - padding = s/2

示例(×32):
  ConvTranspose2d(in_c, out_c, 
                  kernel_size=64,
                  stride=32, 
                  padding=16)
```

In [ ]:
# 转置卷积演示
print("=== 转置卷积 ===")

# 1. 基本转置卷积
def trans_conv_demo(X, K):
    """手动实现转置卷积"""
    h, w = K.shape
    Y = torch.zeros((X.shape[0] + h - 1, X.shape[1] + w - 1))
    for i in range(X.shape[0]):
        for j in range(X.shape[1]):
            Y[i: i + h, j: j + w] += X[i, j] * K
    return Y

X = torch.tensor([[0., 1.], [2., 3.]])
K = torch.tensor([[0., 1.], [2., 3.]])

print("输入 (2×2):")
print(X)
print("\n卷积核 (2×2):")
print(K)

Y_manual = trans_conv_demo(X, K)
print("\n转置卷积输出 (3×3):")
print(Y_manual)

# 2. PyTorch实现
tconv = nn.ConvTranspose2d(1, 1, kernel_size=2, bias=False)
tconv.weight.data = K.reshape(1, 1, 2, 2)
Y_torch = tconv(X.reshape(1, 1, 2, 2))

print("\nPyTorch输出:")
print(Y_torch.squeeze())
print(f"\n两种方法结果相同: {torch.allclose(Y_manual, Y_torch.squeeze())}")

In [ ]:
# 双线性插值初始化
def bilinear_kernel(in_channels, out_channels, kernel_size):
    """生成双线性插值的卷积核"""
    factor = (kernel_size + 1) // 2
    if kernel_size % 2 == 1:
        center = factor - 1
    else:
        center = factor - 0.5
    
    og = (torch.arange(kernel_size).reshape(-1, 1),
          torch.arange(kernel_size).reshape(1, -1))
    filt = (1 - torch.abs(og[0] - center) / factor) * \
           (1 - torch.abs(og[1] - center) / factor)
    
    weight = torch.zeros((in_channels, out_channels, 
                         kernel_size, kernel_size))
    weight[range(in_channels), range(out_channels), :, :] = filt
    return weight

# 测试双线性上采样
print("\n=== 双线性插值上采样 ===")
conv_trans = nn.ConvTranspose2d(3, 3, kernel_size=4, 
                               padding=1, stride=2, bias=False)
conv_trans.weight.data = bilinear_kernel(3, 3, 4)

# 创建测试图像
X_test = torch.rand(1, 3, 10, 10)
Y_test = conv_trans(X_test)

print(f"输入shape: {X_test.shape}")
print(f"输出shape: {Y_test.shape}")
print(f"放大倍数: {Y_test.shape[-1] / X_test.shape[-1]:.0f}×")

print("\n双线性插值特点:")
print("  - 平滑上采样")
print("  - 初始化转置卷积")
print("  - 训练时可继续优化")

### 4.3 FCN架构

In [ ]:
# 简化FCN模型
class SimpleFCN(nn.Module):
    """简化的FCN模型"""
    def __init__(self, num_classes):
        super().__init__()
        # 1. 特征提取(使用预训练ResNet-18)
        pretrained_net = torchvision.models.resnet18(pretrained=False)
        # 去掉全局池化和全连接层
        self.features = nn.Sequential(
            *list(pretrained_net.children())[:-2]
        )
        
        # 2. 1×1卷积调整通道数
        self.conv_cls = nn.Conv2d(512, num_classes, kernel_size=1)
        
        # 3. 转置卷积上采样
        self.transpose_conv = nn.ConvTranspose2d(
            num_classes, num_classes,
            kernel_size=64, padding=16, stride=32
        )
        
    def forward(self, x):
        # x: (batch, 3, H, W)
        x = self.features(x)      # (batch, 512, H/32, W/32)
        x = self.conv_cls(x)      # (batch, num_classes, H/32, W/32)
        x = self.transpose_conv(x) # (batch, num_classes, H, W)
        return x

# 测试FCN
print("=== FCN模型 ===")
num_classes = 21  # VOC
model = SimpleFCN(num_classes)

# 输入图像
X = torch.rand(2, 3, 320, 480)
print(f"输入: {X.shape}  # (batch, RGB, H, W)")

# 前向传播
with torch.no_grad():
    Y = model(X)

print(f"输出: {Y.shape}  # (batch, classes, H, W)")
print(f"\n理解:")
print(f"  - 每个像素: {num_classes}个类别分数")
print(f"  - 空间维度: 保持与输入相同")
print(f"  - 像素级预测: argmax获得类别")

# 获取预测类别
pred = Y.argmax(dim=1)
print(f"\n预测类别图: {pred.shape}  # (batch, H, W)")

### 4.4 FCN变体

**FCN-32s**: 基础版本
```
原图 → 特征 (1/32) → 上采样32× → 预测
问题: 细节丢失
```

**FCN-16s**: 跳跃连接
```
Pool4 (1/16) ─┐
              ├→ 融合 → 上采样16× → 预测
Pool5 (1/32) ─┘
改进: 结合浅层细节
```

**FCN-8s**: 更多跳跃连接
```
Pool3 (1/8)  ─┐
Pool4 (1/16) ─┼→ 融合 → 上采样8× → 预测
Pool5 (1/32) ─┘
效果: 更精细的分割
```

**后续改进**:
- **U-Net**: 对称的编码器-解码器
- **DeepLab**: 空洞卷积
- **PSPNet**: 金字塔池化
- **SegNet**: 编码器-解码器 + 索引

## 5. 小结

### 目标检测

**单阶段 (SSD, YOLO)**:
- ✅ 快速: 实时检测
- ✅ 简单: 端到端训练
- ❌ 精度: 略低于两阶段

**两阶段 (R-CNN系列)**:
- ✅ 精度高: SOTA性能
- ✅ 灵活: 易于扩展(分割等)
- ❌ 较慢: 两阶段计算

**演进**:
```
R-CNN → Fast R-CNN → Faster R-CNN → Mask R-CNN
  ↓         ↓              ↓              ↓
慢速    共享卷积        RPN        实例分割
```

### 语义分割

**FCN贡献**:
- 首个端到端分割网络
- 全卷积架构
- 转置卷积上采样

**关键技术**:
- **转置卷积**: 学习的上采样
- **跳跃连接**: 融合多层特征
- **1×1卷积**: 调整通道数

### 实践建议

**选择模型**:
```
实时检测 → SSD, YOLOv5/v8
高精度检测 → Faster R-CNN, EfficientDet
实例分割 → Mask R-CNN
语义分割 → DeepLabv3+, HRNet
```

**训练技巧**:
1. 预训练权重(ImageNet)
2. 数据增广(必须!)
3. 多尺度训练
4. 学习率调度
5. 类别平衡(Focal Loss)

## 练习

1. **SSD实现**: 完整实现一个简化的SSD模型。

2. **RoI池化**: 手动实现RoI池化,理解其工作原理。

3. **转置卷积**: 实验不同参数的转置卷积,观察输出尺寸变化。

4. **FCN训练**: 在PASCAL VOC2012上训练FCN模型。

5. **模型对比**: 对比SSD和Faster R-CNN在速度和精度上的差异。

6. **数据增广**: 为语义分割设计数据增广策略(注意标签同步)。

7. **可视化**: 可视化FCN不同层的特征图,理解层次表示。

8. **改进**: 为FCN添加跳跃连接,实现FCN-8s。

9. **实例分割**: 了解Mask R-CNN的实现,尝试使用预训练模型。

10. **应用**: 在自己的数据上应用目标检测或语义分割模型。